# CAR-bench Agent Alignment: Odds Ratio Preference Optimization (ORPO) Pipeline (Local Offline Workstation)

This notebook contains a complete training pipeline to align a Large Language Model (LLM) for the CAR-bench voice assistant competition using Odds Ratio Preference Optimization (ORPO).

## Pipeline Architecture
ORPO aligns language models by combining supervised fine-tuning (SFT) and preference learning into a single loss function. Unlike Direct Preference Optimization (DPO), ORPO does not require a separate SFT phase or a reference model. This significantly reduces GPU memory (VRAM) overhead and training time, making it suitable for resource-constrained environments like Google Colab and Kaggle.

## Contents
1. Environment Setup: Install packages and clone repository files.
2. Risk Management: Establish connection keep-alive, persistent storage, and memory recovery safeguards.
3. Data Preparation: Load datasets and parse preference pairs (chosen and rejected turns).
4. Formatting and Tokenization: Apply chat templates for prompt-chosen-rejected mapping.
5. Model Loading: Initialize base weights and configure PEFT/LoRA adapters.
6. Training Configuration: Set training hyperparameters and execute ORPO with automatic checkpoint recovery.
7. Visualization: Plot training losses and reward accuracies.
8. Saving and Merging Model Weights: Save adapters and merge to 16-bit precision weights.
9. Model Quantization and GGUF Export: Quantize to GGUF format.
10. Downloading Results and Checkpoints: Zip and download artifacts.
11. Inference Server Deployment: Launch local server for evaluation.

## CAR-bench Leaderboard & Baselines

Below are the baseline results for frontier and open-source models on the CAR-bench voice assistant tasks. The goal of this alignment pipeline is to fine-tune a smaller, local model (e.g. Qwen2.5-3B-Instruct) to match or exceed these baselines by specializing on the 19 vehicle assistant policies:

| Model | Alignment Method | Avg Pass^3 | Base Pass^3 | Hall. Pass^3 | Disamb. Pass^3 | Source/Platform |
|---|---|---|---|---|---|---|
| Claude Opus 4.6 | RLHF (Frontier) | 0.58 | 0.80 | 0.48 | 0.46 | Anthropic (Official) |
| GPT-5 | RLHF (Frontier) | 0.54 | 0.66 | 0.60 | 0.36 | OpenAI (Official) |
| Gemini 2.5 Pro | RLHF (Frontier) | 0.38 | 0.53 | 0.34 | 0.28 | Google (Official) |
| **Qwen2.5-3B-Instruct (Fine-Tuned)** | **ORPO (This Run)** | **TBD** | **TBD** | **TBD** | **TBD** | **Local / Cloud Run** |
| Qwen3-32B | Base SFT/RLHF | 0.31 | 0.45 | 0.27 | 0.22 | Alibaba (Official) |
| xLAM-2-32B | SFT / DPO | 0.16 | 0.26 | 0.11 | 0.12 | Salesforce (Official) |

## CAR-bench Evaluation Metrics & Strategy
Based on the official CAR-bench evaluation design, the benchmark is significantly more deterministic than it may initially appear. The evaluation is split into:

### 1. Exact / Deterministic Evaluation
* **State-based reward**: Replays ground truth actions to produce expected state hashes, then compares them (`==`) with the agent's state hashes (`base.py:81`).
* **Tool subset reward**: Checks whether the agent called every tool that the ground truth called (`gt_action_names.issubset(performed_action_names)`) (`reward_calculators.py:123`).
* **Tool execution errors**: Checks whether the runtime error list is empty (accumulated during tool execution).
* **Policy (AUT)**: ~200 lines of deterministic checks (tool call names, argument values, vehicle state) (`policy_evaluator.py:116-325`).
* **Success threshold**: Exact float comparison (`1 - 1e-6 <= reward <= 1 + 1e-6`).

### 2. LLM-Prompted Evaluation
* **Policy (LLM)**: ~20 domain-specific policies are checked by sending `(policy, trajectory)` to an LLM judge asking for `{policy_followed: bool, reasoning: str}` (`policy_evaluator.py:88-114`).
* **End conversation**: The user simulator (LLM) decides each turn (`CONTINUE`, `STOP`, `HALLUCINATION_ERROR`, `DISAMBIGUATION_ERROR`, etc.) acting as a quality gate.

> [!NOTE]
> The LLM judge is a supplement, not the core evaluator. Our ORPO/SFT training focuses on mimicking these precise deterministic rules and tool calling behaviors.


## 1. Environment Setup

In [ ]:
# Running in local offline workstation environment
IN_COLAB = False
print("Running on local workstation. Repositories and paths are assumed local.")

In [ ]:
# Ensure you have installed these packages in your local environment:
# pip install transformers peft trl datasets accelerate bitsandbytes huggingface_hub protobuf matplotlib seaborn
# pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print("Please verify packages are pre-installed in your virtual environment.")

In [ ]:
import os
import sys
import json
import torch
import random

# Establish seeds for training reproducibility
random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## 2. Risk Management & Persistent Storage

To prevent training failures due to timeouts, RAM overflows, or cloud VM disconnects, we establish the following safeguards:
1. Keep-Alive Script: Javascript to prevent notebook inactivity disconnects.
2. Google Drive Mounting: Saves checkpoints directly to persistent storage (Google Drive or Kaggle outputs) rather than transient VM storage.
3. Memory Cleanup: Garbage collection and GPU cache purge to avoid Out-Of-Memory (OOM) errors.

### Keep-Alive Script
Paste the following JavaScript code into the browser developer console (F12) to simulate clicks and prevent timeouts during long training sessions:

```javascript
function KeepAlive() {
    console.log("Simulating interaction to keep session active...");
    var connectButton = document.querySelector("colab-connect-button");
    if (connectButton) connectButton.click();
}
setInterval(KeepAlive, 60000);
```

In [ ]:
# Local output directory configured for local PC
PERSISTENT_DIR = "./outputs"
os.makedirs(PERSISTENT_DIR, exist_ok=True)
print(f"Target checkpoints directory configured locally: {PERSISTENT_DIR}")

In [ ]:
import gc

def clear_gpu_memory():
    """Purges system garbage collection and releases cached PyTorch CUDA memory."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("Released GPU cache memory.")

clear_gpu_memory()

## 3. Data Preparation

We load training data and construct preference pairs containing a `prompt` (conversation history), `chosen` (correct turn response), and `rejected` (suboptimal or incorrect turn response).

In [ ]:
if os.path.exists(dpo_dataset_path):
    print(f"Loading preference dataset from {dpo_dataset_path}...")
    dataset = load_dataset("json", data_files=dpo_dataset_path, split="train")
else:
    print("Dataset file not found. Parsing raw logs to extract preference pairs...")
    wiki_file = "third_party/car-bench/car_bench/envs/car_voice_assistant/wiki.md"
    if not os.path.exists(wiki_file):
        wiki_file = "car-bench-ijcai/third_party/car-bench/car_bench/envs/car_voice_assistant/wiki.md"
        if not os.path.exists(wiki_file):
            wiki_file = "car-bench-ijcai-vsf/third_party/car-bench/car_bench/envs/car_voice_assistant/wiki.md"
    dataset = extract_dpo_preference_dataset("output", wiki_file)

print(f"Preference dataset loaded. Total samples: {len(dataset)}")

# Split dataset into train/evaluation sets (90% train, 10% eval) for metric comparison
if len(dataset) > 0:
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_dataset = dataset_split["train"]
    eval_dataset = dataset_split["test"]
    print(f"Dataset split: {len(train_dataset)} train samples, {len(eval_dataset)} evaluation samples")
else:
    train_dataset = dataset
    eval_dataset = dataset


## 4. Formatting and Tokenization

We apply the tokenizer chat template to format the prompt history and responses into a single string for training.

In [ ]:
# RESTRICTED ORDER OF IMPORTS - Unsloth must be imported BEFORE transformers/peft/trl
try:
    import unsloth
except ImportError:
    pass

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_orpo_data(examples):
    formatted_prompts = []
    formatted_chosen = []
    formatted_rejected = []
    
    for prompt, chosen, rejected in zip(examples["prompt"], examples["chosen"], examples["rejected"]):
        # Format prompt history (ends with assistant start token)
        p_text = tokenizer.apply_chat_template(prompt, tokenize=False, add_generation_prompt=True)
        
        # Format assistant turns
        c_text = tokenizer.apply_chat_template(chosen, tokenize=False, add_generation_prompt=False)
        r_text = tokenizer.apply_chat_template(rejected, tokenize=False, add_generation_prompt=False)
        
        # Remove prompt prefix if tokenizer duplicates it
        if c_text.startswith(p_text):
            c_text = c_text[len(p_text):]
        if r_text.startswith(p_text):
            r_text = r_text[len(p_text):]
            
        formatted_prompts.append(p_text)
        formatted_chosen.append(c_text)
        formatted_rejected.append(r_text)
        
    return {
        "prompt": formatted_prompts,
        "chosen": formatted_chosen,
        "rejected": formatted_rejected
    }

if len(train_dataset) > 0:
    formatted_train_dataset = train_dataset.map(format_orpo_data, batched=True)
    formatted_eval_dataset = eval_dataset.map(format_orpo_data, batched=True) if len(eval_dataset) > 0 else None
    print("Example Formatted DPO Sample (Train):")
    print(f"Prompt:\n{formatted_train_dataset[0]['prompt'][:300]}...\n")
    print(f"Chosen:\n{formatted_train_dataset[0]['chosen']}\n")
    print(f"Rejected:\n{formatted_train_dataset[0]['rejected']}")
else:
    formatted_train_dataset = train_dataset
    formatted_eval_dataset = eval_dataset
    print("No data available to format.")


## 5. Model Loading and PEFT Configuration

In [ ]:
# Pre-loading clean memory check
clear_gpu_memory()

USE_UNSLOTH = True
MAX_SEQ_LENGTH = 2048

if USE_UNSLOTH:
    print("Loading model via Unsloth...")
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        trust_remote_code=True
    )
    
    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=3407,
    )
else:
    print("Loading model via Standard HF PEFT...")
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=False,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    model = prepare_model_for_kbit_training(model)
    
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)

print("Model initialized. Trainable parameters:")
model.print_trainable_parameters()

## 6. Training Configuration and ORPO Execution

We define the training arguments using the persistent storage directory configured in the Risk Management section. We also check for existing checkpoints to automatically resume execution if a server restart occurs.

In [ ]:
from trl import ORPOTrainer, ORPOConfig

# Check for existing checkpoints to resume training automatically
resume_checkpoint = None
if os.path.exists(PERSISTENT_DIR):
    checkpoints = [os.path.join(PERSISTENT_DIR, d) for d in os.listdir(PERSISTENT_DIR) if d.startswith("checkpoint-")]
    if checkpoints:
        # Sort checkpoints by step iteration count
        checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
        resume_checkpoint = checkpoints[-1]
        print(f"Found existing checkpoint: {resume_checkpoint}. Resuming from this step.")

orpo_config = ORPOConfig(
    output_dir=PERSISTENT_DIR,
    learning_rate=5e-6,      # Recommended range: 5e-6 to 1e-5
    beta=0.1,                # Odds Ratio loss multiplier (lambda)
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    optim="paged_adamw_8bit",
    eval_strategy="epoch",             # Evaluate at the end of each epoch
    save_strategy="epoch",             # Save checkpoint at the end of each epoch
    save_total_limit=1,                # Save ONLY the single best checkpoint
    load_best_model_at_end=True,       # Load the best model at the end of training
    metric_for_best_model="eval_loss", # Use validation loss to compare checkpoints
    greater_is_better=False,
    logging_steps=5,
    max_length=MAX_SEQ_LENGTH,
    max_prompt_length=MAX_SEQ_LENGTH // 2,
    report_to="none",
    average_tokens_across_devices=False
)

trainer = ORPOTrainer(
    model=model,
    args=orpo_config,
    train_dataset=formatted_train_dataset,
    eval_dataset=formatted_eval_dataset,
    tokenizer=tokenizer
)

if len(formatted_train_dataset) > 0:
    print("Starting ORPO training...")
    try:
        trainer_stats = trainer.train(resume_from_checkpoint=resume_checkpoint)
        print("Training completed successfully!")
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print("\nCUDA Out Of Memory encountered. Suggested mitigation steps:")
            print("1. Set MAX_SEQ_LENGTH to 1024 or 512.")
            print("2. Set per_device_train_batch_size to 1.")
            print("3. Execute 'clear_gpu_memory()' and restart execution cell.")
        raise e
else:
    print("Dataset is empty. Add preference samples to dataset before running.")


## 7. Training Metrics Visualization

We plot SFT loss, Odds Ratio loss, and accuracy metrics collected during training iterations.

In [ ]:
import matplotlib.pyplot as plt

if hasattr(trainer, "state") and trainer.state.log_history:
    history = trainer.state.log_history
    
    train_steps = []
    train_losses = []
    eval_steps = []
    eval_losses = []
    eval_accuracies = []
    
    for log in history:
        step = log.get("step")
        if "loss" in log:  # Training log
            train_steps.append(step)
            train_losses.append(log["loss"])
        if "eval_loss" in log:  # Evaluation log
            eval_steps.append(step)
            eval_losses.append(log["eval_loss"])
            if "eval_rewards/accuracies" in log:
                eval_accuracies.append(log["eval_rewards/accuracies"])
                
    plt.figure(figsize=(15, 5))
    
    # Loss Comparison Plot
    plt.subplot(1, 2, 1)
    if train_losses:
        plt.plot(train_steps, train_losses, label="Train Loss", color="red", marker="o")
    if eval_losses:
        plt.plot(eval_steps, eval_losses, label="Validation Loss", color="blue", marker="s")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title("Train vs Validation Loss Comparison")
    plt.legend()
    plt.grid(True)
    
    # Accuracy Curve Plot
    plt.subplot(1, 2, 2)
    if eval_accuracies:
        plt.plot(eval_steps, eval_accuracies, label="Validation Reward Accuracy", color="green", marker="^")
        plt.xlabel("Step")
        plt.ylabel("Accuracy")
        plt.title("Preference Reward Accuracy on Validation Set")
        plt.legend()
        plt.grid(True)
    else:
        plt.text(0.5, 0.5, "No Validation Accuracy recorded", ha="center", va="center")
        
    plt.tight_layout()
    plt.show()
else:
    print("No log history found to plot.")

## 8. Saving and Merging Model Weights

We save the LoRA adapter weights and merge them into the base model weights to produce a 16-bit precision checkpoint for deployment.

In [ ]:
ADAPTER_PATH = os.path.join(PERSISTENT_DIR, "lora_adapter")
MERGED_PATH = os.path.join(PERSISTENT_DIR, "merged_model")

# Save training adapter checkpoints
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f"LoRA adapter weights saved to {ADAPTER_PATH}")

# Save merged weights (16-bit precision)
if USE_UNSLOTH:
    print("Saving merged 16-bit model weights via Unsloth...")
    model.save_pretrained_merged(MERGED_PATH, tokenizer, save_method="merged_16bit")
    print(f"Merged model saved to {MERGED_PATH}")
else:
    print("Merging standard HF PEFT model weights...")
    from peft import PeftModel
    from transformers import AutoModelForCausalLM
    
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16 if not torch.cuda.is_bf16_supported() else torch.bfloat16,
        device_map="cpu",
        trust_remote_code=True
    )
    peft_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
    merged_model = peft_model.merge_and_unload()
    
    merged_model.save_pretrained(MERGED_PATH)
    tokenizer.save_pretrained(MERGED_PATH)
    print(f"Merged model saved to {MERGED_PATH}")

## 9. Model Quantization and GGUF Export

When using Unsloth, weights can be exported directly to GGUF format for memory-efficient local deployment.

In [ ]:
if USE_UNSLOTH:
    GGUF_PATH = os.path.join(PERSISTENT_DIR, "merged_q4_k_m")
    print("Exporting model to Q4_K_M GGUF format...")
    try:
        model.save_pretrained_merged(GGUF_PATH, tokenizer, save_method="quantized_gguf", quantization_method="q4_k_m")
        print(f"GGUF model exported to {GGUF_PATH}-unsloth.Q4_K_M.gguf")
    except Exception as e:
        print(f"Export to GGUF failed: {e}")
        
    # Provide download option in Colab environments
else:
    print("GGUF export is only supported when running with Unsloth.")

## 10. Downloading Results and Checkpoints

This section provides helper functions to compress (zip) and download model checkpoints, adapter files, or full training outputs. This is especially useful in Google Colab (triggers browser download) and Kaggle (saves zip file to outputs for easy download via sidebar).

In [ ]:
import shutil
import os

def zip_and_download(folder_path, output_filename):
    """Zips a folder and triggers download in Colab, or creates a downloadable file in Kaggle/local."""
    if not os.path.exists(folder_path):
        print(f"Folder '{folder_path}' does not exist. Skipping compression.")
        return
        
    zip_path = f"{output_filename}.zip"
    print(f"Zipping '{folder_path}' into '{zip_path}'...")
    
    # Zip the folder
    shutil.make_archive(output_filename, 'zip', folder_path)
    file_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"Created: {zip_path} ({file_size_mb:.2f} MB)")
    
# Choose what you want to download:
# Option 1: Download LoRA adapter only (recommended, small size)
zip_and_download(ADAPTER_PATH, "lora_adapter_results")

# Option 2: Download full training outputs/checkpoints folder (large size)
# zip_and_download(PERSISTENT_DIR, "all_training_checkpoints")

## 11. Inference Server Deployment

Serve the model locally using vLLM to integrate with the evaluation harness:

```bash
pip install vllm
python -m vllm.entrypoints.openai.api_server \
    --model ./merged_model \
    --port 8000 \
    --dtype float16 \
    --gpu-memory-utilization 0.90
```

Configure environmental variables in `.env`:
```env
AGENT_LLM=openai/car-bench-llm
OPENAI_API_BASE=http://localhost:8000/v1
OPENAI_API_KEY=local-dummy-key
```

Execute the evaluation harness:
```bash
uv run car-bench-run --scenario scenarios/track_1_agent_under_test/local_smoke.toml
```